# YOLOv8s traffic sign baseline on Kaggle

Runs training, optional validation threshold tuning, and diagnostics with the prepared street-grouped split. Select a Kaggle GPU accelerator before running.

In [ ]:
from pathlib import Path
import os

DATA_ROOT = Path('/kaggle/input/datasets/ducmonday/zalo-traffic-sign/za_traffic_2020')
REPO_DIR = Path('/kaggle/working/AIT3001.Group6.traffic.sign.detection')
assert (DATA_ROOT / 'traffic_train' / 'annotation.csv').is_file(), f'Cannot find annotation.csv under {DATA_ROOT}'
assert (DATA_ROOT / 'traffic_train' / 'images').is_dir(), f'Cannot find images under {DATA_ROOT}'
print('Dataset:', DATA_ROOT)

In [ ]:
# Kaggle Internet must be enabled for cloning and installing missing packages.
%cd /kaggle/working
!git clone https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git
%cd /kaggle/working/AIT3001.Group6.traffic.sign.detection
!python -m pip install -r requirements.txt

## Baseline settings and threshold tuning

Train first, then run the optional tuning cell with the exact W&B run name printed on the run page. By default it selects NMS IoU using mAP@0.5:0.95, then confidence using F1. Tuning writes CSV results and `best_thresholds.json` under `outputs/tuning/`; use those values in the diagnostics command afterward.

In [ ]:
# Optional: store WANDB_API_KEY in Kaggle Secrets before enabling online tracking.
TRACKING_MODE = 'online'  # set to offline if no W&B secret is configured
if TRACKING_MODE == 'online':
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')

In [ ]:
# Prepare grouped split/labels only; image resizing happens online inside each model pipeline.
!python train.py prepare_only=true data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs

In [ ]:
# YOLOv8s baseline, no augmentation. Override epochs/batch/device as needed for the Kaggle GPU.
!python train.py model=yolov8 data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs augmentation.enabled=false model.weights=yolov8s.pt model.imgsz=640 model.epochs=50 model.batch_size=16 model.device=0 evaluation.inference.score_threshold=0.001 evaluation.inference.nms_iou=0.7 tracking.mode={TRACKING_MODE}

## Tune inference thresholds (optional)

Copy the exact run name from W&B. This reuses the prepared validation split and does not train the model again.

In [ ]:
# Replace with the exact W&B run name for the trained checkpoint.
TUNING_RUN_NAME = 'your-exact-wandb-run-name'
!python tune.py "tuning.run_name={TUNING_RUN_NAME}" data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs

## Optional Faster R-CNN run

Run this separately if the GPU session has enough time and memory. It uses the same prepared street split and 640×640 online letterbox geometry; model normalization remains in torchvision.

In [ ]:
# !python train.py model=fasterrcnn data.root="{DATA_ROOT}" data.prepared_dir=/kaggle/working/prepared_traffic_signs augmentation.enabled=false model.imgsz=640 model.epochs=50 model.batch_size=2 model.device=0 tracking.mode={TRACKING_MODE}

In [ ]:
# Package local outputs so they can be downloaded from the Kaggle notebook output panel.
!cd /kaggle/working && zip -qr traffic_sign_outputs.zip AIT3001.Group6.traffic.sign.detection/outputs